# 07. Principled Uncertainty Gating & Single-Flow Streaming Benchmarks

**Project:** High-Throughput Network Intrusion Detection with Decoupled XAI  
**Objective:** Evaluate and visualize Normalized Shannon Entropy gating, Split-Conformal Prediction Sets, and single-flow streaming latency ($\text{batch}=1$ with cache warmup).

---

## 1. Mathematical Formulation & Architecture

### A. Normalized Shannon Entropy Gating
Traditional cascaded NIDS architectures rely on arbitrary heuristic cutoffs ($p_{\text{low}} < P(y=1|x) < p_{\text{high}}$). We replace heuristic thresholds with Normalized Shannon Entropy:

$$H(x) = -\frac{1}{\ln 2} \sum_{c \in \{0, 1\}} P(y=c \mid x) \ln P(y=c \mid x)$$

Where $H(x) \in [0, 1]$. Flows exceeding the calibrated threshold $\tau_H = 0.80$ represent boundary ambiguity and are escalated to Tier 2.

### B. Split-Conformal Prediction Sets
To provide distribution-free finite-sample risk coverage guarantees, we compute non-conformity scores on a calibration split:

$$s_i = 1 - \hat{P}_{\text{T1}}(y_i \mid x_i)$$

$$\hat{q}_\alpha = \text{Quantile}\left( \{s_i\}_{i=1}^n; \, \frac{\lceil (n+1)(1-\alpha) \rceil}{n} \right)$$

Prediction sets $\Gamma_\alpha(x)$ guarantee $P(Y \in \Gamma_\alpha(X)) \ge 1 - \alpha$. Ambiguous flows ($|\Gamma_\alpha(x)| = 2$) and out-of-distribution flows ($|\Gamma_\alpha(x)| = 0$) are escalated to Tier 2 deep sequence triage.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.cascade_controller import CascadedNIDSController

print("[✓] Modules imported successfully.")

## 2. Benchmark Model Setup & Calibration

In [ ]:
np.random.seed(42)
n_samples = 20000
X = np.random.randn(n_samples, 30).astype(np.float32)
y = np.random.binomial(1, 0.25, size=n_samples).astype(np.int32)
X[y == 1, :5] += 2.0

split1, split2 = int(n_samples * 0.60), int(n_samples * 0.80)
X_train, y_train = X[:split1], y[:split1]
X_cal, y_cal = X[split1:split2], y[split1:split2]
X_test, y_test = X[split2:], y[split2:]

class FastQuantizedTreeModel:
    def __init__(self, feature_dim=30):
        self.feature_dim = feature_dim
        self.weights = np.random.randn(feature_dim).astype(np.float32) * 0.4
        self.bias = -0.5

    def fit(self, X, y):
        X_sub = X[:5000, :self.feature_dim]
        y_sub = y[:5000]
        self.weights = np.linalg.solve(X_sub.T @ X_sub + 1e-2 * np.eye(self.feature_dim), X_sub.T @ y_sub)

    def predict_proba(self, X):
        X_mat = np.asarray(X, dtype=np.float32)
        if X_mat.ndim == 1:
            X_mat = X_mat.reshape(1, -1)
        dots = np.dot(X_mat[:, :self.feature_dim], self.weights[:min(self.feature_dim, X_mat.shape[1])]) + self.bias
        dots = np.clip(dots, -40.0, 40.0)
        p1 = 1.0 / (1.0 + np.exp(-dots))
        probs = np.zeros((len(X_mat), 2), dtype=np.float32)
        probs[:, 1] = np.clip(p1, 0.0001, 0.9999)
        probs[:, 0] = 1.0 - probs[:, 1]
        return probs

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(np.int32)

tier1 = FastQuantizedTreeModel(feature_dim=30)
tier1.fit(X_train, y_train)
print(f"[+] Model fitted on {len(X_train):,} training records.")

## 3. Normalized Shannon Entropy & Split-Conformal Prediction Evaluation

In [ ]:
controller_entropy = CascadedNIDSController(tier1, gating_mode="entropy", entropy_threshold=0.80)
res_ent = controller_entropy.evaluate_traffic_stream(X_test, y_true=y_test)
print(f"[Entropy Gating] Resolved Inline: {res_ent['tier1_resolved_pct']:.2f}% | Escalated: {res_ent['tier2_escalated_pct']:.2f}%")

controller_conf = CascadedNIDSController(tier1, gating_mode="conformal", conformal_alpha=0.05)
q = controller_conf.calibrate_conformal_quantile(X_cal, y_cal)
res_conf = controller_conf.evaluate_traffic_stream(X_test, y_true=y_test)
print(f"[Conformal Gating (alpha=0.05)] Calibrated q: {q:.4f} | Resolved: {res_conf['tier1_resolved_pct']:.2f}% | Escalated: {res_conf['tier2_escalated_pct']:.2f}%")

## 4. Single-Flow Streaming Latency Profile (batch=1 with Cache Warmup)

In [ ]:
streaming_stats = controller_conf.benchmark_streaming_latency(X_test, n_warmup=1000, n_eval=min(5000, len(X_test)))
print("=" * 55)
print("HARDWARE TESTBED STREAMING LATENCY PROFILE (batch=1)")
print("=" * 55)
print(f"  Mean Latency       : {streaming_stats['mean_us']:.2f} us/flow")
print(f"  P50 Median Latency : {streaming_stats['p50_us']:.2f} us/flow")
print(f"  P90 Latency        : {streaming_stats['p90_us']:.2f} us/flow")
print(f"  P99 Tail Latency   : {streaming_stats['p99_us']:.2f} us/flow")
print(f"  Streaming Rate     : {streaming_stats['streaming_throughput_fps']:,.0f} flows/sec")
print("=" * 55)